# Parte III — Auditoría temporal y cobertura

Esta auditoría verifica las entradas actuales y los cortes antes de entrenar. Usa pandas y PyArrow sobre columnas acotadas de datos ya procesados; no publica tablas ni modifica fuentes. Los conteos no son métricas de un recomendador. Fechas Parquet se interpretan como instantes UTC y se convierten a America/Lima, coherente con el contrato. Los releases compilados actuales no reconstruyen versiones históricas: incluso las fechas anteriores al corte no prueban disponibilidad real.

In [1]:
from pathlib import Path
import hashlib
import pandas as pd
import pyarrow.dataset as ds
import pyarrow.parquet as pq
from IPython.display import display

ROOT = Path.cwd()
if not (ROOT / "docs").exists():
    ROOT = ROOT.parent
assert (ROOT / "docs/ContratoDatos.md").exists(), "Ejecute desde la raíz o notebooks/"
paths = {name: ROOT / "data/processed" / name for name in
         ("fact_award", "fact_award_supplier", "entity_supplier_year")}
MAX_ROWS = 500_000
MAX_BYTES = 256 * 1024**2
for name, path in paths.items():
    assert (path / "_SUCCESS").exists(), f"Falta _SUCCESS: {name}"
    files = sorted(path.rglob("*.parquet"))
    assert files, f"Sin Parquet: {name}"
    size = sum(f.stat().st_size for f in files)
    rows = sum(pq.ParquetFile(f).metadata.num_rows for f in files)
    assert size <= MAX_BYTES and rows <= MAX_ROWS, f"Límite excedido: {name}"
    h = hashlib.sha256()
    for file in files:
        h.update(file.relative_to(path).as_posix().encode() + b"\0")
        with file.open("rb") as stream:
            for block in iter(lambda: stream.read(1024 * 1024), b""):
                h.update(block)
    print(f"{name}: filas={rows:,}, bytes={size:,}, huella de contenido y rutas={h.hexdigest()}")

def load(name, columns):
    return ds.dataset(paths[name], format="parquet", partitioning="hive",
                      exclude_invalid_files=True).to_table(columns=columns).to_pandas()

def lima(values):
    return pd.to_datetime(values, utc=True).dt.tz_convert("America/Lima")

def blank(values):
    return values.isna() | values.astype("string").str.strip().eq("").fillna(False)


fact_award: filas=128,395, bytes=23,352,125, huella de contenido y rutas=e6b41d9886ab732eb63d0fd3017da4033b6b1bf7c0a51d068a3d4efcd13a4a71
fact_award_supplier: filas=128,395, bytes=10,754,866, huella de contenido y rutas=1d0f325d710241bdec50b8967ebaf927751cec42df932069025e1390978b072b
entity_supplier_year: filas=108,924, bytes=2,264,484, huella de contenido y rutas=391857eadeea7b327f8d9bb22c20ee1a0803cab391e2ce5178f0bf7ba322c487


## Claves, fechas y conciliación

La clave de adjudicación es `process_ocid × award_id`; una adjudicación puede tener varios proveedores. Comparamos atributos comunes y los conteos de la tabla anual, incluyendo moneda para respetar su grano. Las razones de exclusión pueden solaparse.

In [2]:
key = ["process_ocid", "award_id"]
s = load("fact_award_supplier", key + ["supplier_id", "entity_id", "award_date", "award_year", "award_currency", "release_id", "source_year"])
a = load("fact_award", key + ["entity_id", "award_date", "release_date", "published_date", "release_id", "source_year"])
y = load("entity_supplier_year", ["entity_id", "supplier_id", "award_year", "award_currency", "award_count"])
assert not a.duplicated(key).any(), "Claves de adjudicación duplicadas"
assert not s.duplicated(key + ["supplier_id"]).any(), "Claves adjudicación-proveedor duplicadas"
j = s.merge(a, on=key, how="left", suffixes=("_s", "_a"), validate="many_to_one", indicator=True)
assert j["_merge"].eq("both").all(), "Adjudicaciones sin correspondencia"
for col in ["entity_id", "award_date", "release_id", "source_year"]:
    left, right = j[col + "_s"], j[col + "_a"]
    assert (left.eq(right) | (left.isna() & right.isna())).all(), f"Diferencias en {col}"
s["local_date"] = lima(s["award_date"])
assert (s["local_date"].dt.year.eq(s["award_year"]) | (s["local_date"].isna() & s["award_year"].isna())).all(), "Año no coincide en Lima"
missing = pd.DataFrame({"filas": [len(s), blank(s.entity_id).sum(), blank(s.supplier_id).sum(), s.award_date.isna().sum(),
     (blank(s.entity_id) | blank(s.supplier_id) | s.award_date.isna()).sum()]},
     index=["total", "entidad ausente", "proveedor ausente", "fecha ausente", "unión de exclusiones"])
display(missing)
eligible = s.loc[~(blank(s.entity_id) | blank(s.supplier_id) | s.award_date.isna())].copy()
eligible["currency_key"] = eligible.award_currency.fillna("<MISSING>")
y["currency_key"] = y.award_currency.fillna("<MISSING>")
grain = ["entity_id", "supplier_id", "award_year", "currency_key"]
expected = eligible.groupby(grain, dropna=False).size().rename("expected").reset_index()
actual = y.groupby(grain, dropna=False).award_count.sum().rename("actual").reset_index()
comparison = expected.merge(actual, on=grain, how="outer", validate="one_to_one").fillna({"expected": 0, "actual": 0})
assert comparison.expected.eq(comparison.actual).all(), "Interacciones anuales no concilian"
print(f"Claves y atributos conciliados; {len(comparison):,} grupos anuales coinciden.")

,filas
total,128395
entidad ausente,0
proveedor ausente,0
fecha ausente,35
unión de exclusiones,35


Claves y atributos conciliados; 108,924 grupos anuales coinciden.


## Cortes, pares nuevos y casos fríos

Se cuentan pares distintos por período. Las categorías de entidad y proveedor frío pueden solaparse; no se suman. El denominador de cobertura son todos los pares nuevos observados, no todas las adjudicaciones.

In [3]:
results = []
for year in (2023, 2024):
    train = eligible.loc[eligible.award_year < year]
    future = eligible.loc[eligible.award_year == year]
    tp = set(zip(train.entity_id, train.supplier_id))
    fp = set(zip(future.entity_id, future.supplier_id))
    entities = set(train.entity_id); suppliers = set(train.supplier_id)
    new = fp - tp
    evaluable = {(e, p) for e, p in new if e in entities and p in suppliers}
    results.append({"corte": year, "filas entrenamiento": len(train), "filas período": len(future),
        "entidades entrenamiento": len(entities), "proveedores entrenamiento": len(suppliers),
        "pares entrenamiento": len(tp), "pares nuevos": len(new),
        "pares repetidos": len(fp & tp), "nuevos evaluables": len(evaluable),
        "entidades evaluables": len({e for e, p in evaluable}),
        "entidad fría": sum(e not in entities for e, p in new),
        "proveedor frío": sum(p not in suppliers for e, p in new),
        "ambos fríos": sum(e not in entities and p not in suppliers for e, p in new),
        "cobertura nuevos": len(evaluable) / len(new) if new else float("nan")})
coverage = pd.DataFrame(results).set_index("corte")
display(coverage.T)

corte,2023,2024
filas entrenamiento,46636.000000,88395.000000
filas período,41759.000000,39965.000000
entidades entrenamiento,2691.000000,2883.000000
proveedores entrenamiento,28878.000000,47563.000000
pares entrenamiento,39780.000000,71396.000000
pares nuevos,31616.000000,26570.000000
pares repetidos,2682.000000,6522.000000
nuevos evaluables,9806.000000,10038.000000
entidades evaluables,1843.000000,1879.000000
entidad fría,603.000000,180.000000


## Fechas de publicación y disponibilidad

Para los eventos de entrenamiento de cada corte se cuenta si `release_date` y `published_date` son anteriores, posteriores o ausentes. Una fecha anterior no acredita la versión disponible al corte. Se compara cada campo de forma independiente; no se suman sus categorías entre sí.

In [4]:
j["local_date"] = lima(j["award_date_s"])
for field in ("release_date", "published_date"):
    j[field + "_local"] = lima(j[field])
rows = []
for year in (2023, 2024):
    cutoff = pd.Timestamp(f"{year}-01-01", tz="America/Lima")
    valid = ~(blank(j.entity_id_s) | blank(j.supplier_id) | j.local_date.isna())
    tr = j.loc[valid & (j.local_date < cutoff)]
    for field in ("release_date", "published_date"):
        dates = tr[field + "_local"]
        rows.append({"corte": year, "campo": field, "filas entrenamiento": len(tr),
                     "anterior al corte": int((dates < cutoff).sum()),
                     "desde el corte": int((dates >= cutoff).sum()),
                     "ausente": int(dates.isna().sum())})
display(pd.DataFrame(rows).set_index(["corte", "campo"]))
print("Estas fechas describen los releases actuales; no prueban disponibilidad histórica ni eliminan fuga de información.")

filas entrenamiento  anterior al corte  desde el corte  \
corte campo                                                                    
2023  release_date                  46636                  0           46636   
      published_date                46636                  0           46636   
2024  release_date                  88395              36922           51473   
      published_date                88395              36922           51473   

                      ausente  
corte campo                    
2023  release_date          0  
      published_date        0  
2024  release_date          0  
      published_date        0

Estas fechas describen los releases actuales; no prueban disponibilidad histórica ni eliminan fuga de información.


## Decisión de avance

La siguiente etapa debe mantener el alcance retrospectivo y registrar cobertura de casos fríos. Los conteos y huellas anteriores corresponden a las entradas presentes al ejecutar este notebook; no acreditan por sí solos la procedencia completa hasta los archivos originales. Para reproducir la cadena completa se requiere reconstruirla desde el manifiesto verificado.